# [EDA] 고객 단위 집계 피처 탐색 — 2026-06-29

**목적**: `run_customer_pipeline.py`로 생성된 집계 테이블을 탐색한다.

**입력 파일**:
- `data/processed/customer_features_all_customers.csv` — 전체 고객 (20,000명)
- `data/processed/customer_features_us_customers.csv` — US 고객 (3,648명)

**분석 항목**:
1. 기본 검사 (shape, dtypes, null, duplicates)
2. 수치형 피처 분포 (recency, count, spend)
3. 카테고리 피처 분포 (top_view_category, top_purchase_category)
4. 전체 vs US 비교

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import seaborn as sns

# 한국어 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# [VS Code]
DATA_DIR = '../data/processed'

# [Colab] Colab에서 실행 시 위 DATA_DIR 주석 처리 후 아래 두 줄 주석 해제
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/why-they-leave/retail-clickstream-analysis/processed'

In [ ]:
df_all = pd.read_csv(f'{DATA_DIR}/customer_features_all_customers.csv')
df_us  = pd.read_csv(f'{DATA_DIR}/customer_features_us_customers.csv')

print(f'전체: {df_all.shape}  |  US-only: {df_us.shape}')

## 1. 기본 검사

In [ ]:
df_all.dtypes

In [ ]:
# 결측치 비율
null_rate = df_all.isna().mean().round(3) * 100
null_rate.to_frame('null_%')

In [ ]:
# customer_id 중복 확인
print('전체 중복:', df_all['customer_id'].duplicated().sum())
print('US 중복:  ', df_us['customer_id'].duplicated().sum())

In [ ]:
df_all.describe().T

## 2. 수치형 피처 분포

In [ ]:
num_cols = [
    'recency_session_days', 'recency_order_days',
    'session_count', 'page_view_count', 'add_to_cart_count',
    'order_count', 'total_spend', 'avg_order_value'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df_all[col].dropna()
    axes[i].hist(data, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('count')

plt.suptitle('수치형 피처 분포 (전체 고객)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 주문 없는 고객 비율
no_order_rate = (df_all['order_count'] == 0).mean() * 100
no_session_rate = (df_all['session_count'] == 0).mean() * 100
print(f'주문 없는 고객: {no_order_rate:.1f}%')
print(f'세션 없는 고객: {no_session_rate:.1f}%')

## 3. 카테고리 피처 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# top_view_category
view_counts = df_all['top_view_category'].value_counts()
axes[0].barh(view_counts.index, view_counts.values, color='steelblue')
axes[0].set_title('top_view_category 분포 (전체)')
axes[0].set_xlabel('고객 수')
axes[0].invert_yaxis()

# top_purchase_category
purchase_counts = df_all['top_purchase_category'].value_counts()
axes[1].barh(purchase_counts.index, purchase_counts.values, color='coral')
axes[1].set_title('top_purchase_category 분포 (전체)')
axes[1].set_xlabel('고객 수')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# 조회 vs 구매 카테고리 일치율
both = df_all.dropna(subset=['top_view_category', 'top_purchase_category'])
match_rate = (both['top_view_category'] == both['top_purchase_category']).mean() * 100
print(f'조회 카테고리 == 구매 카테고리 일치율: {match_rate:.1f}%')

## 4. 전체 vs US 비교

In [ ]:
compare_cols = ['session_count', 'page_view_count', 'order_count', 'total_spend', 'avg_order_value']

comparison = pd.DataFrame({
    '전체 (mean)': df_all[compare_cols].mean(),
    'US (mean)':   df_us[compare_cols].mean(),
    '전체 (median)': df_all[compare_cols].median(),
    'US (median)':   df_us[compare_cols].median(),
}).round(2)

comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# top_purchase_category 전체 vs US 비교
all_cat = df_all['top_purchase_category'].value_counts(normalize=True) * 100
us_cat  = df_us['top_purchase_category'].value_counts(normalize=True) * 100

cats = all_cat.index.tolist()
x = np.arange(len(cats))
w = 0.35

axes[0].bar(x - w/2, all_cat.reindex(cats).fillna(0), w, label='전체', color='steelblue')
axes[0].bar(x + w/2, us_cat.reindex(cats).fillna(0),  w, label='US',   color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cats, rotation=45, ha='right')
axes[0].set_title('구매 카테고리 비율 비교 (%)')
axes[0].set_ylabel('%')
axes[0].legend()

# total_spend 분포 비교
axes[1].hist(df_all['total_spend'].clip(upper=df_all['total_spend'].quantile(0.99)),
             bins=50, alpha=0.5, label='전체', color='steelblue')
axes[1].hist(df_us['total_spend'].clip(upper=df_us['total_spend'].quantile(0.99)),
             bins=50, alpha=0.5, label='US',   color='coral')
axes[1].set_title('total_spend 분포 비교 (상위 1% 제거)')
axes[1].set_xlabel('total_spend (USD)')
axes[1].set_ylabel('count')
axes[1].legend()

plt.tight_layout()
plt.show()